# Indexing Strategies: Granular vs. Coarse/Multi-Vector Chunking

This notebook compares RAG indexing strategies over Wikivoyage travel pages.

**Paradigm 1 — Naive chunk-and-embed:** split a document into chunks and embed each chunk directly. The text you search on is the same text you get back.

**Paradigm 2 — Multi-vector retrieval:** decouple the *searchable* representation (small, semantically clean) from the *returned* content (fuller, more useful context), using LangChain's `MultiVectorRetriever` with a vectorstore (search) + docstore (content) pair linked by a shared id. Two variants are explored:
- **Variant A:** search on an LLM-generated summary of a large chunk, return the original large chunk.
- **Variant B:** search on a small precise chunk, return a stitched-together window of neighboring chunks (no LLM cost).

| | Paradigm 1 (naive) | Paradigm 2A (summary) | Paradigm 2B (window) |
|---|---|---|---|
| Search representation | full chunk | LLM summary | small chunk |
| Returned content | same as search | large coarse chunk | prev+cur+next window |
| Extra indexing cost | none | LLM call per chunk | none |
| Storage | single vectorstore | vectorstore + docstore | vectorstore + docstore |

In [1]:
from langchain_chroma import Chroma
import sys 
from langchain_huggingface import HuggingFaceEmbeddings
sys.path.append("..")
from llm_model import llm
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

cornwall_granular_collection = Chroma(collection_name="cornwall_granular",
    embedding_function=embeddings_model)

cornwall_granular_collection.reset_collection()

cornwall_coarse_collection = Chroma(collection_name="cornwall_coarse",
    embedding_function=embeddings_model)
cornwall_coarse_collection.reset_collection()

/Users/bahloulia/Downloads/agentic_software/RAG-Applications/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paradigm 1: Naive chunk-and-embed

Fetch a single page (Cornwall), split it on its own HTML structure (`h1`/`h2` sections) with `HTMLSectionSplitter`, and embed each section chunk directly into `cornwall_granular_collection`. `similarity_search` matches the query against those same chunk embeddings and returns them as-is — one vector per chunk, no indirection between what's searched and what's returned.

In [ ]:
from langchain_community.document_loaders import AsyncHtmlLoader

# Wikimedia's crawler policy blocks/serves a robots-notice page instead of
# the article for requests with no identifying User-Agent - same fix as
# the wikipedia loader in the other notebook.
wikimedia_header_template = {
    "User-Agent": "RAG-Applications/1.0 (amine.bahlouli1993@gmail.com)"}

destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(
    destination_url, header_template=wikimedia_header_template)
docs = html_loader.load()
from langchain_text_splitters import HTMLSectionSplitter

headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string)
        all_chunks.extend(temp_chunks) 

    return all_chunks

granular_chunks = split_docs_into_granular_chunks(docs)
cornwall_granular_collection.add_documents(documents=granular_chunks)
results = cornwall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

## Paradigm 2: Multi-vector retrieval

Build a `MultiVectorRetriever` where:
- a **vectorstore** holds a small, search-optimized representation of each chunk, and
- a **docstore** (`InMemoryByteStore`) holds the fuller content actually handed to the LLM, keyed by a shared `doc_id`.

A similarity search against the vectorstore resolves — via the shared id — to the richer content in the docstore. The next two sections try two different ways of generating that search-optimized representation.

In [9]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Variant A: LLM-generated summaries as the search key

Split each page into large (3000-char) coarse chunks, then run each coarse chunk through an LLM summarization chain. The **summary** is embedded into `summaries_collection` (the search representation); the **original coarse chunk** is stored in the docstore under the same id. Queries match against the concise summary but retrieve the full original chunk. This costs one LLM call per chunk at indexing time.

In [3]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import sys
import uuid

sys.path.append("..")
from llm_model import llm

# TODO: fill in the actual set of UK destination pages you want to index
uk_destination_urls = [
    "https://en.wikivoyage.org/wiki/Cornwall",
    "https://en.wikivoyage.org/wiki/London",
    "https://en.wikivoyage.org/wiki/Edinburgh",
    "https://en.wikivoyage.org/wiki/Bath",
]

# Wikimedia's crawler policy blocks/serves a robots-notice page instead of
# the article for requests with no identifying User-Agent.
wikimedia_header_template = {
    "User-Agent": "RAG-Applications/1.0 (amine.bahlouli1993@gmail.com)"}

html2text_transformer = Html2TextTransformer()

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000)

summaries_collection = Chroma(
    collection_name="uk_summaries",
    embedding_function=embeddings_model,
)

summaries_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)



summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Summarize the following document:\n\n{document}")
    | llm
    | StrOutputParser())

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url, header_template=wikimedia_header_template)
    html_docs =  html_loader.load()
    text_docs = html2text_transformer.transform_documents(
        html_docs)

    coarse_chunks = parent_splitter.split_documents(
        text_docs)

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks):

        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text =  summarization_chain.invoke(
            coarse_chunk)
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))

retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

print(summary_docs_only[0])

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.42it/s]


KeyboardInterrupt: 

Sanity check: retrieve via the summary-indexed multi-vector retriever and confirm we get back the original (non-summarized) chunk content.

In [3]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
print(retrieved_docs[0])

page_content='The Night Riviera sleeper service operates between London and Penzance six
nights a week (Su-F). Trains feature cabins and an on-board lounge.

Great Western Railway also operates regular slow trains between Bristol Temple
Meads, Exeter St Davids, Plymouth and Penzance, calling in Cornwall at
**Saltash** , **St Germans** , **Liskeard** , **Bodmin Parkway** ,
**Lostwithiel** , **Par** , **St Austell** , **Truro** , **Redruth** ,
**Camborne** , **Hayle** , **St Erth** and **Penzance**.

Some destinations lie on local lines, so journeys may require a change of
trains at:

  * Plymouth for **Calstock** and **Gunnislake**
  * Liskeard for **Looe**
  * Par for **Newquay**
  * Truro for **Penryn** and **Falmouth**
  * St Erth or Penzance for **St Ives**

A small number of CrossCountry services call at **Liskeard** , **Bodmin
Parkway** , **St Austell** , **Truro** , **Redruth** , **St Erth** and
**Penzance** from destinations throughout the UK, including: Edinburgh,
Newcastle upo

### Variant B: sliding-window expansion (no LLM)

Split each page into small (500-char) granular chunks and embed each one **directly** — no summarization needed, since the chunk is already small enough to embed precisely. But the docstore entry for each chunk is an **expanded window**: previous chunk + this chunk + next chunk concatenated together. The match is precise (small chunk), while the returned content has surrounding context stitched in. Cheaper than Variant A (no LLM calls at indexing time), at the cost of some duplicated boundary text between neighboring windows.

In [4]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
from langchain_community.document_transformers import Html2TextTransformer
from langchain_core.documents import Document
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500)
uk_destination_urls = [
    "https://en.wikivoyage.org/wiki/Cornwall",
    "https://en.wikivoyage.org/wiki/London",
    "https://en.wikivoyage.org/wiki/Edinburgh",
    "https://en.wikivoyage.org/wiki/Bath",
]

# Wikimedia's crawler policy blocks/serves a robots-notice page instead of
# the article for requests with no identifying User-Agent.
wikimedia_header_template = {
    "User-Agent": "RAG-Applications/1.0 (amine.bahlouli1993@gmail.com)"}

html2text_transformer = Html2TextTransformer()
granular_chunks_collection = Chroma(
    collection_name="uk_granular_chunks",
    embedding_function=embeddings_model
)

granular_chunks_collection.reset_collection()

expanded_chunk_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url, header_template=wikimedia_header_template)
    html_docs =  html_loader.load()
    text_docs = html2text_transformer.transform_documents(
        html_docs)

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs)

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks):

        this_chunk_num = i
        previous_chunk_num = i - 1 if i > 0 else None
        next_chunk_num = i + 1 if i < len(granular_chunks) - 1 else None

        expanded_chunk_text = ""
        if previous_chunk_num is not None:
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content
        expanded_chunk_text += "\n"

        if next_chunk_num is not None:
            expanded_chunk_text += granular_chunks[  
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4())
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text)

        expanded_chunk_store_item = (expanded_chunk_id, 
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks)
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.04it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.60it/s]


Ingesting https://en.wikivoyage.org/wiki/London


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.10it/s]


Ingesting https://en.wikivoyage.org/wiki/Edinburgh


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.24it/s]


Ingesting https://en.wikivoyage.org/wiki/Bath


Sanity check: retrieve via the window-expanded multi-vector retriever and confirm the returned content includes the stitched-together neighboring chunks.

In [6]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
print(retrieved_docs[0])

page_content='_For other places with the same name, seeCornwall (disambiguation)._
**Cornwall** (Cornish: _Kernow_) is a county in the southwest of England.
Lying west of Devon from which it is separated by the River Tamar, Cornwall is
one of the more isolated and distinctive parts of the United Kingdom but is
also one of its most popular with holidaymakers. Its relatively warm climate,
long coastline, amazing scenery, and diverse Celtic heritage (combined with
tales of smuggling, pirates and King Arthur!) go only part of the way to
explaining its appeal.
The biomes that house the Eden Project, near St. Austell, Mid-Cornwall.
'
